<a href="https://colab.research.google.com/github/Ridd1kulusC0d3r/OSINTIA/blob/main/notebooks/00_nucleo_sem_gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 00 · O núcleo OSINTIA sem GPU

**OSINTIA** · modelos de IA para OSINT, reconstruídos e auditáveis · [repositório](https://github.com/Ridd1kulusC0d3r/OSINTIA)

Antes de treinar qualquer modelo, aprenda a **medir** se ele merece confiança. Este notebook roda só com a biblioteca padrão do Python e usa o caso de demonstração (perfis que imitam o fictício *Banco Aurora*).

| | |
|---|---|
| **Hardware** | CPU (não precisa de GPU) |
| **Tempo estimado** | 2 minutos |
| **Como rodar** | Menu *Ambiente de execução* → *Executar tudo* |

> ⚖️ Uso ético: só fontes abertas, finalidade legítima e respeito à LGPD. Os dados de demonstração são **fictícios**. Leia `docs/METODOLOGIA.md`.

In [ ]:
# Baixa o OSINTIA (núcleo em Python puro: calibração, auditoria, roteamento, custódia)
!git clone -q https://github.com/Ridd1kulusC0d3r/OSINTIA.git /content/OSINTIA 2>/dev/null || git -C /content/OSINTIA pull -q
import sys
sys.path.insert(0, "/content/OSINTIA")
from osintia import auditoria, calibracao, custodia, dados, esquemas, extracao, roteamento
print("OSINTIA pronto")

## 1. As perguntas de decisão
Um modelo de decisão não escreve texto: ele escolhe entre opções definidas antes. Veja o esquema de triagem de perfis e valide contra os limites da API (64 perguntas, 255 opções, 10 níveis).

In [ ]:
import json
esquema = esquemas.TRIAGEM_PERFIL
print(json.dumps(esquema, indent=2, ensure_ascii=False))
erros, avisos = esquemas.validar(esquema)
print("erros:", erros, "| avisos:", avisos)
print("catálogo disponível:", list(esquemas.CATALOGO))

## 2. O caso de demonstração
Cada linha tem `state` (o perfil em texto), `questions`, `gold` (média de 3 respostas de um **professor** simulado) e `rotulo_humano` (a verificação de um analista). O professor tem um vício proposital: **todo perfil com selo de verificado vira `legitimo`**.

In [ ]:
linhas = dados.carregar_jsonl("/content/OSINTIA/dados/demo/triagem_perfis.jsonl")
print(len(linhas), "perfis")
print(linhas[0]["state"])
print(json.loads(linhas[0]["gold"])["perfil"])
print("humano:", linhas[0]["rotulo_humano"])

## 3. O teto do professor
Sorteamos uma 4ª resposta do professor (fora do gold) e medimos quanto ela concorda com o gold. Esse é o máximo **honesto**: um aluno que passa disso está decorando o professor.

In [ ]:
gold   = dados.rotulos(linhas, "perfil")
humano = [l["rotulo_humano"]["perfil"] for l in linhas]
extra  = [l["amostra_professor_extra"]["perfil"] for l in linhas]

teto = auditoria.teto_do_professor([extra], gold)
print(f"teto do professor:  {teto:.3f}")
print(f"gold x humano:      {auditoria.concordancia(gold, humano):.3f}")
print(f"kappa gold x humano: {auditoria.kappa_cohen(gold, humano):.3f}")
print()
print(auditoria.imprimir_matriz(humano, gold))

## 4. Calibração perfeita, verdade imperfeita
Tratamos as probabilidades do gold como se fossem saídas de um modelo. Contra o próprio gold, a calibração é ótima. Contra o humano, aparece o erro herdado.

In [ ]:
probs = [json.loads(l["gold"])["perfil"]["probabilities"] for l in linhas]
print("vs gold:  ", calibracao.relatorio(probs, gold))
print("vs humano:", calibracao.relatorio(probs, humano))

herdados = auditoria.erros_herdados(gold, gold, humano)
print(f"\n{len(herdados)} erros que a métrica oficial chamaria de acerto. Exemplo:\n")
print(linhas[herdados[0]]["state"])

## 5. Temperature scaling
Quando o modelo é superconfiante, dividir os logits por uma temperatura `T > 1` corrige a confiança sem mudar a decisão. É o que o `FastDecisionModel.calibrate` faz por baixo (reconstrução em `docs/ENGENHARIA-REVERSA.md`).

In [ ]:
import random
r = random.Random(7)
# modelo de brinquedo superconfiante: diz entre 70% e 99%, mas acerta só 80% do que diz
OPCOES = ("legitimo", "imitador", "bot")
brinquedo, verdade = [], []
for h in humano:
    conf = r.uniform(0.70, 0.99)
    escolha = h if r.random() < conf * 0.80 else r.choice([o for o in OPCOES if o != h])
    p = {o: (1 - conf) / 2 for o in OPCOES}
    p[escolha] = conf
    brinquedo.append(p); verdade.append(h)

antes = calibracao.relatorio(brinquedo, verdade)
T = calibracao.ajustar_temperatura(brinquedo, verdade)
depois = calibracao.relatorio([calibracao.aplicar_temperatura(p, T) for p in brinquedo], verdade)
print("antes: ", antes)
print(f"T = {T:.2f}")
print("depois:", depois)

## 6. Humano no circuito
Escolha o limiar olhando cobertura (quanto o modelo decide sozinho) contra acurácia automática.

In [ ]:
calibrado = [calibracao.aplicar_temperatura(p, T) for p in brinquedo]
for linha in roteamento.varrer_limiares(calibrado, verdade, limiares=(0.4, 0.5, 0.6, 0.7)):
    print(linha)
print(roteamento.rotear({"legitimo": 0.12, "imitador": 0.81, "bot": 0.07}))

## 7. Extração determinística e LGPD
Regras com dígito verificador para CPF, CNPJ, telefone e IP. No notebook 02 elas confirmam o que o GLiNER propõe.

In [ ]:
texto = "Chame @suporte_bancoaurora no (31) 99999-0000, envie CPF 529.982.247-25 para estorno@aurora-falso.example"
for e in extracao.extrair(texto):
    print(e)
print(extracao.mascarar(texto))

## 8. Cadeia de custódia
Todo resultado vira um evento com hash SHA-256 encadeado. Se alguém editar o arquivo depois, a verificação aponta onde.

In [ ]:
import os
os.makedirs("/content/saidas", exist_ok=True)
dados.salvar_jsonl([{"id": l["id"], "decisao": g} for l, g in zip(linhas, gold)], "/content/saidas/decisoes.jsonl")
man = "/content/saidas/manifesto.jsonl"
if os.path.exists(man): os.remove(man)
custodia.registrar(man, "carga_dataset", "/content/OSINTIA/dados/demo/triagem_perfis.jsonl")
custodia.registrar(man, "decisoes", "/content/saidas/decisoes.jsonl", {"teto_professor": round(teto, 4)})
print("problemas:", custodia.verificar(man) or "nenhum, cadeia íntegra")

---
### Próximos passos
- `docs/ENGENHARIA-REVERSA.md` explica como cada peça foi reconstruída.
- `docs/AUDITORIA-DE-ROTULOS.md` mostra por que acurácia alta pode esconder erro herdado.
- Abra uma *issue* no repositório se algo quebrar no Colab: as bibliotecas de IA mudam rápido.

*OSINTIA · Deivison Lourenço · MIT*